# mon_projet: waste-sorting metrics and natural-language assistant

**Scope:** week 8, site Montreuil only (the only data available).
Everything runs on the Python standard library.

**Confirmed decisions**
- Recycling rate = `recycle_kg / collecte_kg`; resale rate = `revendu_kg / collecte_kg` (confirmed definition).
- pandas is not allowed: standard library only.

**Convention**
- Any figure not present in `donnees.csv` is marked `[TO CONFIRM]`.
- The API key is read from the `ANTHROPIC_API_KEY` environment variable, never written in the notebook.

In [ ]:
import csv
import json
import os
import urllib.request

CSV_PATH = "donnees.csv"
API_URL = "https://api.anthropic.com/v1/messages"
MODEL = "claude-sonnet-5-5"
OUTPUT_COLUMNS = ["revendu_kg", "recycle_kg", "dechetterie_kg", "attente_kg"]

## 1. Load the data

In [ ]:
def load_rows(path):
    """Load the CSV file and return a list of dicts with numeric fields."""
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    for row in rows:
        row["semaine"] = int(row["semaine"])
        row["collecte_kg"] = float(row["collecte_kg"])
        for col in OUTPUT_COLUMNS:
            row[col] = float(row[col])
    return rows


rows = load_rows(CSV_PATH)
print(f"{len(rows)} rows loaded")

## 2. Compute the rates

In [ ]:
def check_balance(row):
    """Return the gap between collected weight and the sum of outputs (kg)."""
    return row["collecte_kg"] - sum(row[c] for c in OUTPUT_COLUMNS)


def rate(part, total):
    """Return part/total as a percentage rounded to 1 decimal, or None."""
    if total == 0:
        return None
    return round(100 * part / total, 1)


def compute_metrics(rows):
    """Add recycling_rate_pct and resale_rate_pct to each row."""
    for row in rows:
        row["recycling_rate_pct"] = rate(row["recycle_kg"], row["collecte_kg"])
        row["resale_rate_pct"] = rate(row["revendu_kg"], row["collecte_kg"])
    return rows


def totals(rows):
    """Return overall totals and rates across all rows."""
    collected = sum(r["collecte_kg"] for r in rows)
    recycled = sum(r["recycle_kg"] for r in rows)
    resold = sum(r["revendu_kg"] for r in rows)
    return {
        "collected_kg": collected,
        "recycling_rate_pct": rate(recycled, collected),
        "resale_rate_pct": rate(resold, collected),
    }

In [ ]:
def print_table(rows):
    """Print a per-category table of rates and balance checks."""
    print(f"{'Category':<10}{'Collected kg':>13}{'Recycling %':>13}{'Resale %':>10}{'Balance gap kg':>16}")
    for r in rows:
        print(f"{r['categorie']:<10}{r['collecte_kg']:>13.0f}"
              f"{r['recycling_rate_pct']:>13}{r['resale_rate_pct']:>10}"
              f"{check_balance(r):>16.0f}")


rows = compute_metrics(rows)
print_table(rows)
print("\nTotals:", totals(rows))

## 3. Natural-language assistant

The assistant only sees the data below and must answer `[TO CONFIRM]` when a figure is missing.

In [ ]:
def build_context(rows):
    """Serialize the data and metrics as JSON text for the assistant."""
    return json.dumps({"rows": rows, "totals": totals(rows)}, ensure_ascii=False)


def ask_claude(question, context):
    """Send a question plus data context to the API and return the answer."""
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        raise RuntimeError("Set the ANTHROPIC_API_KEY environment variable.")
    system = (
        "You answer questions about waste-sorting data. Use only the JSON "
        "provided. If a figure is not in the data, write [TO CONFIRM] "
        "instead of guessing. Answer in the language of the question."
    )
    body = json.dumps({
        "model": MODEL,
        "max_tokens": 1024,
        "system": system,
        "messages": [{
            "role": "user",
            "content": f"Data:\n{context}\n\nQuestion: {question}",
        }],
    }).encode("utf-8")
    req = urllib.request.Request(API_URL, data=body, headers={
        "x-api-key": key,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json",
    })
    with urllib.request.urlopen(req) as resp:
        return json.load(resp)["content"][0]["text"]

In [ ]:
question = "Quelle catégorie a le meilleur taux de recyclage, et laquelle a le plus faible ?"

if os.environ.get("ANTHROPIC_API_KEY"):
    print(ask_claude(question, build_context(rows)))
else:
    print("ANTHROPIC_API_KEY is not set: define it in your environment, then re-run this cell.")